# Engine-matrix CPU harness (Kaggle pinned image)

The fake-server harnesses of the EMS (SGLang 0.5.21) and VMS (vLLM 0.32.0) engine-matrix notebooks, plus the VMS #58807 bad-ignore-list case, on the same image as the GPU notebooks. Kit: `foysalemonshanto/q27-engine-cpu-kit` (q27x_engine_cpu_kit_32f8142b3272.bin).


In [ ]:
# Engine-matrix CPU harness on Kaggle (pinned image = the GPU notebooks'): EMS (SGLang) + VMS (vLLM 0.32.0) harnesses
import hashlib, io, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
KIT_NAME, KIT_SHA = "q27x_engine_cpu_kit_32f8142b3272.bin", "32f8142b327217e3f38815b5936ac7503d0edd3e534f4e7c81929dfd23210369"
RUNS = [["ems_mtpx", "e14src/harness/harness_ems_nb.py", "Q27-XHIGH-EMS_MTPX_ENGINE_MATRIX_SGLANG.ipynb", {"HARNESS_EMS_PART": "MTPX"}]]
WH = "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"
OUT = Path("/kaggle/working"); K = Path("/tmp/engine_kit"); T0 = time.time()
REPORT = {"kit": KIT_NAME, "python": sys.version, "nb_sha": {}, "harness": {}}


def save():
    (OUT / "cpu_harness_report.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")


def find(name, sha):
    for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name, "*/*/*/*/" + name):
        for f in sorted(Path("/kaggle/input").glob(pat)):
            b = f.read_bytes()
            if hashlib.sha256(b).hexdigest() == sha:
                return b
    raise FileNotFoundError(f"{name} ({sha[:12]}) not under /kaggle/input: attach foysalemonshanto/q27-engine-cpu-kit")


shutil.rmtree(K, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(find(KIT_NAME, KIT_SHA))) as z:
    z.extractall(K)
man = json.loads((K / "MANIFEST.json").read_text())
bad = [k for k, h in man.items() if hashlib.sha256((K / k).read_bytes()).hexdigest() != h]
assert not bad, bad
for nb in sorted((K / "nb").glob("*.ipynb")):
    REPORT["nb_sha"][nb.name] = hashlib.sha256(nb.read_bytes()).hexdigest()
print("kit OK:", len(man), "files |", sys.version.split()[0], "|", REPORT["nb_sha"], flush=True)
# the GPU notebooks' runtime pins (their install cell): transformers 4.57.6 + huggingface-hub 0.36.2 from the S49 wheelhouse
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH,
                    "transformers==4.57.6", "huggingface-hub==0.36.2"], capture_output=True, text=True)
REPORT["pins"] = {"rc": r.returncode, "tail": (r.stdout + r.stderr)[-800:]}
print("pins:", r.returncode, flush=True)
ENV = dict(os.environ, E14_SGL_SRC=str(K / "sgl_ref"), E14_FIXTURES_DIR=str(K / "fixtures"), PYTHONIOENCODING="utf-8",
           E14_HARNESS_DIR=str(K / "e14src/harness"), E1105_HARNESS_DIR=str(K / "e1105"), E15P_HARNESS_DIR=str(K / "e15p"),
           EMS_PAYLOAD_DIR=str(K / "payload"), VMS_REDHAT_INDEX=str(K / "redhat_index.json"))
for name, script, nb, extra in RUNS:
    env = dict(ENV, E14_HARNESS_ROOT=f"/tmp/engine_h_{name}", E15P_MODE="commit", **extra)
    t = time.time()
    try:
        r = subprocess.run([sys.executable, str(K / script), str(K / "nb" / nb), "0.03"],
                           cwd=K / "e14src/harness", env=env, capture_output=True, text=True, timeout=3600)
        out, rc = r.stdout + r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        out, rc = (e.stdout or "") if isinstance(e.stdout, str) else "", "timeout"
    (OUT / f"harness_{name}.log").write_text(out, encoding="utf-8")
    lines = out.splitlines()
    REPORT["harness"][name] = {"rc": rc, "sec": round(time.time() - t), "ok": rc == 0 and "CHECKS PASS" in out,
                               "summary": [x for x in lines if "CHECKS PASS" in x or "failed of" in x],
                               "fails": [x[:1500] for x in lines if x.startswith("FAIL")],
                               "cell_sec": [x for x in lines if x.startswith("cell seconds")],
                               "traceback_tail": out[-3000:] if "Traceback" in out else ""}
    save()
    print(name, "->", REPORT["harness"][name]["summary"], REPORT["harness"][name]["fails"][:6], flush=True)
REPORT["sec"] = round(time.time() - T0)
REPORT["all_ok"] = all(v["ok"] for v in REPORT["harness"].values())
save()
print("ALL OK" if REPORT["all_ok"] else "NOT OK", "| total", REPORT["sec"], "s", flush=True)
